# Base_Metal_download_v4
KOMIS 자원 가격 수집 → investar 적재 (엑셀 저장은 선택)

**저장 테이블 (investar, 표준 규격 STD-1)**

| 테이블 | 내용 | 키 |
|---|---|---|
| `kr_komis_mineral_price_info` | 가격 시리즈 설명표 (광종, 분류, 가격기준, 주기, 단위) | series_id |
| `kr_komis_mineral_price` | 가격 최신본 (시리즈 x 날짜 1행) | series_id, period_end |
| `kr_komis_mineral_price_vintage` | 값이 처음 들어오거나 바뀐 이력 | id |

- `series_id` = `광종코드_가격기준코드` (예: `MNRL0008_501` = 동, 가격기준 501). 한 광종에 가격기준이 여러 개여도 섞이지 않습니다.
- STD-1: 소문자 snake_case / utf8mb4_general_ci / `period_end` + `freq`(D,W,M) / 수집 시각 KST / 이력은 `_vintage` / 단위·통화는 설명표에 둔다.

**v4 변경**: 시리즈별로 DB에 있는 마지막 날짜를 보고 자동으로 시작 연도를 정함 (DB에 없는 광종은 자동으로 장기 백필).
요청을 1년 단위로 나눠 받고, 연도별 수신 건수를 출력. KOMIS 제공 기간 진단 셀(5-0) 추가.

**사용 순서**: 1~4번 셀 실행 → (선택) 5-0 제공 기간 진단 → 5번 수집·적재 → 6번 검증

## 1. 설정 및 접속

In [1]:
import os
import sys
import json
import time
from pathlib import Path
from datetime import date, datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
from sqlalchemy import text


def setup_universal_paths():
    """현재 폴더에서 상위로 올라가며 DATA 폴더를 찾아 sys.path에 추가."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print("프로젝트 루트:", parent)
            return parent
    raise FileNotFoundError("DATA 폴더를 찾지 못했습니다. investment_strategy 아래에서 실행하세요.")


ROOT = setup_universal_paths()
import config

engine = config.get_engine(config.get_db_info())
KST = timezone(timedelta(hours=9))

T_INFO = "kr_komis_mineral_price_info"
T_DATA = "kr_komis_mineral_price"
T_VIN = "kr_komis_mineral_price_vintage"

# 엑셀 저장 경로 (SAVE_EXCEL = True일 때만 사용)
USER_HOME = Path(r"C:\Users\82108")
SUBPATH = Path("OneDrive") / "INVESTMENT" / "지표상회_복제" / "result_files"


def default_output_dir():
    """1) 환경변수 KOMIS_OUTPUT_DIR  2) USER_HOME/SUBPATH  3) 현재 사용자 홈/SUBPATH"""
    env = os.environ.get("KOMIS_OUTPUT_DIR")
    if env:
        return Path(env)
    if USER_HOME.is_dir():
        return USER_HOME / SUBPATH
    fallback = Path.home() / SUBPATH
    print(f"[안내] {USER_HOME} 를 찾을 수 없습니다. {Path.home()} 기준으로 저장합니다.")
    return fallback

print("DB 엔진 준비 완료")

프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
DB 엔진 준비 완료


## 2. KOMIS 수집기

In [2]:
AVG_TO_FREQ = {"DAY": "D", "WEEK": "W", "MONTH": "M"}


class KOMISCrawler:
    def __init__(self):
        self.base_url = "https://www.komis.or.kr"
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "application/json, text/javascript, */*; q=0.01",
            "Accept-Language": "ko-KR,ko;q=0.9",
            "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
            "X-Requested-With": "XMLHttpRequest",
            "Origin": "https://www.komis.or.kr",
        })

        # avg: 카테고리 기본 평균옵션. DAY 데이터가 없으면 WEEK → MONTH 순으로 자동 재시도
        self.categories = {
            "HP001": {"name": "비철금속", "url": "/Komis/RsrcPrice/BaseMetals", "avg": "DAY",
                      "minerals": {"니켈": "MNRL0002", "동": "MNRL0008", "아연": "MNRL0023",
                                   "알루미늄": "MNRL0009", "연": "MNRL0022", "주석": "MNRL0016"}},
            "HP002": {"name": "희소금속", "url": "/Komis/RsrcPrice/MinorMetals", "avg": "DAY",
                      "minerals": {"리튬": "MNRL0001", "코발트": "MNRL0003", "망간": "MNRL0004",
                                   "크롬": "MNRL0021", "몰리브덴": "MNRL0012", "바나듐": "MNRL0013",
                                   "텅스텐": "MNRL0018", "티타늄": "MNRL0017", "안티모니": "MNRL0019",
                                   "니오븀": "MNRL0007", "규소": "MNRL0010", "마그네슘": "MNRL0011",
                                   "흑연": "MNRL0005", "갈륨": "MNRL0024", "게르마늄": "MNRL0035",
                                   "인듐": "MNRL0025", "셀레늄": "MNRL0029", "탄탈륨": "MNRL0026",
                                   "지르코늄": "MNRL0027", "스트론튬": "MNRL0028"}},
            "HP003": {"name": "철광석및에너지", "url": "/Komis/RsrcPrice/IronOre", "avg": "WEEK",
                      "minerals": {"유연탄": "MNRL0032"}},   # 철광석 코드는 웹사이트 확인 필요
            "HP004": {"name": "기타광석", "url": "/Komis/RsrcPrice/EtcMnrl", "avg": "DAY",
                      "minerals": {"금": "MNRL0046", "은": "MNRL0047", "백금": "MNRL0014",
                                   "팔라듐": "MNRL0015", "우라늄": "MNRL0031"}},
        }

    def update_referer(self, category_url):
        self.session.headers["Referer"] = self.base_url + category_url

    def get_price_criteria(self, mineral_code, hp000="HP001"):
        """광종별 가격기준 조회. 반환: (코드, 이름, 원본 dict)"""
        url = f"{self.base_url}/Komis/RsrcPrice/ajax/getMnrlPriceCrtr"
        try:
            r = self.session.post(url, data={"HP000": hp000, "mnrkndUnqCd": mineral_code}, timeout=10)
            if r.status_code == 200:
                rows = r.json().get("data") or []
                if rows:
                    item = rows[0]
                    name = None
                    for k in ("cdNm", "cdKeyNm", "cdValue", "cdVal", "codeNm", "prcCrtrNm"):
                        if item.get(k):
                            name = str(item[k])
                            break
                    return str(item.get("cdKey", "") or ""), name, item
        except Exception as e:
            print(f"  가격기준 조회 오류: {e}")
        return "", None, {}

    def get_price_data(self, mineral_code, price_criteria, start_year, end_year, avg_option, hp000):
        """가격 데이터 원본 JSON 조회. 실패 시 None."""
        url = f"{self.base_url}/Komis/RsrcPrice/ajax/getMnrlPrcByMnrkndUnqCd"
        params = {
            "HP000": hp000, "srchMnrkndUnqCd": mineral_code, "srchPrcCrtr": price_criteria,
            "srchAvgOpt": avg_option, "srchField": "year",
            "srchStartDate": str(start_year), "srchEndDate": str(end_year),
            "srchCompareMnrkndUnqCd": "", "srchComparePrcCrtr": "", "lmeInvt": "N",
        }
        try:
            r = self.session.post(url, data=params, timeout=30)
            if r.status_code == 200:
                return r.json()
            print(f"  데이터 조회 실패: HTTP {r.status_code}")
        except Exception as e:
            print(f"  데이터 조회 오류: {e}")
        return None

    def fetch_series(self, mineral_name, hp000, start_year, end_year, avg_option=None,
                     chunk_years=1, verbose=True):
        """한 광종의 가격을 표준 형태 DataFrame으로 반환.
        반환: (df[crtrYmd, price, change, change_pct], meta dict) 또는 (None, meta)
        """
        cat = self.categories[hp000]
        mineral_code = cat["minerals"][mineral_name]
        self.update_referer(cat["url"])
        crit_code, crit_name, crit_raw = self.get_price_criteria(mineral_code, hp000)

        meta = {"mineral_code": mineral_code, "mineral_name": mineral_name,
                "category_code": hp000, "category_name": cat["name"],
                "price_criteria_code": crit_code, "price_criteria_name": crit_name,
                "criteria_raw": crit_raw, "info_raw": {}, "avg_option": None}

        tries = [avg_option or cat["avg"]]
        for alt in ("WEEK", "MONTH"):
            if alt not in tries:
                tries.append(alt)

        for avg in tries:
            frames, got = [], []
            for y0 in range(start_year, end_year + 1, chunk_years):
                y1 = min(y0 + chunk_years - 1, end_year)
                res = self.get_price_data(mineral_code, crit_code, y0, y1, avg, hp000)
                time.sleep(0.5)
                if not res:
                    continue
                info = (res.get("dataAvg") or {}).get("INFO") or {}
                if info:
                    meta["info_raw"] = info
                rows = ((res.get("data") or {}).get("defaultMnrl")) or []
                if rows:
                    frames.append(pd.DataFrame(rows))
                    got.append(f"{y0}:{len(rows)}")
            if verbose and got:
                print(f"  {mineral_name} {avg} 연도별 수신: " + ", ".join(got))
            if frames:
                df = pd.concat(frames, ignore_index=True)
                need = ["crtrYmd", "cmercPrc", "flctnPrc", "flctnPrcnt"]
                for c in need:
                    if c not in df.columns:
                        df[c] = np.nan
                df = df[need].rename(columns={"cmercPrc": "price", "flctnPrc": "change",
                                              "flctnPrcnt": "change_pct"})
                for c in ("price", "change", "change_pct"):
                    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False)
                                          .str.replace("%", "", regex=False), errors="coerce")
                df["crtrYmd"] = df["crtrYmd"].astype(str).str.strip()
                df = df.drop_duplicates("crtrYmd", keep="last")
                meta["avg_option"] = avg
                return df, meta
            print(f"  {mineral_name}: {avg} 데이터 없음")
        return None, meta


def probe_history(mineral_name, hp000="HP001", first_year=1990, avg_options=("DAY", "WEEK", "MONTH")):
    """KOMIS가 한 광종을 몇 년부터 제공하는지 평균옵션별로 확인 (DB 저장 없음)."""
    crawler = KOMISCrawler()
    cat = crawler.categories[hp000]
    code = cat["minerals"][mineral_name]
    crawler.update_referer(cat["url"])
    crit, crit_name, _ = crawler.get_price_criteria(code, hp000)
    out = []
    for avg in avg_options:
        for y in range(first_year, date.today().year + 1):
            res = crawler.get_price_data(code, crit, y, y, avg, hp000)
            time.sleep(0.3)
            rows = (((res or {}).get("data") or {}).get("defaultMnrl")) or []
            if rows:
                d = sorted(str(r.get("crtrYmd")) for r in rows)
                out.append((avg, y, len(rows), d[0], d[-1]))
    res = pd.DataFrame(out, columns=["avg_option", "year", "n_rows", "first", "last"])
    if res.empty:
        print(f"{mineral_name}: 수신 데이터 없음")
        return res
    print(f"{mineral_name} (가격기준 {crit} {crit_name or ''}) 평균옵션별 제공 시작 연도:")
    for avg, g in res.groupby("avg_option", sort=False):
        print(f"  {avg:<5}: {g.year.min()} ~ {g.year.max()}  ({int(g.n_rows.sum())}건)")
    return res


def save_excel(df, meta, start_year, end_year, output_dir=None):
    """기존 v2와 같은 형식으로 엑셀 저장 (선택 기능)."""
    output_dir = Path(output_dir or default_output_dir())
    output_dir.mkdir(parents=True, exist_ok=True)
    out = df.rename(columns={"crtrYmd": "기준일", "price": "기준가격",
                             "change": "등락가", "change_pct": "등락율(%)"}).copy()
    if meta["avg_option"] in ("DAY", "WEEK"):
        out["기준일"] = pd.to_datetime(out["기준일"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    else:
        out["기준일"] = out["기준일"].str[:4] + "-" + out["기준일"].str[4:6]
    fn = (f"{meta['category_name']}_{meta['mineral_name']}_{meta['avg_option']}_"
          f"{start_year}_{end_year}.xlsx")
    out.to_excel(output_dir / fn, index=False, engine="openpyxl")
    return output_dir / fn

print("수집기 준비 완료")

수집기 준비 완료


## 3. 테이블 생성 (없을 때만)

In [3]:
_OPTS = "ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci"

DDL = [
f"""
CREATE TABLE IF NOT EXISTS {T_INFO} (
    series_id           VARCHAR(50)  NOT NULL COMMENT '광종코드_가격기준코드',
    mineral_code        VARCHAR(20)  NOT NULL COMMENT 'KOMIS 광종코드 (MNRL....)',
    mineral_name        VARCHAR(50)  NULL     COMMENT '광종명(한글)',
    category_code       VARCHAR(10)  NULL     COMMENT 'HP001~HP004',
    category_name       VARCHAR(50)  NULL     COMMENT '비철금속, 희소금속 등',
    price_criteria_code VARCHAR(20)  NULL     COMMENT 'KOMIS 가격기준 코드',
    price_criteria_name VARCHAR(100) NULL,
    freq                CHAR(1)      NOT NULL COMMENT 'D,W,M',
    unit                VARCHAR(50)  NULL,
    currency            CHAR(3)      NULL     COMMENT 'ISO 4217',
    criteria_raw        TEXT         NULL     COMMENT '가격기준 응답 원본(JSON)',
    info_raw            TEXT         NULL     COMMENT '광종 정보 응답 원본(JSON)',
    backfill_from_year  SMALLINT     NULL     COMMENT '과거 백필을 시도한 가장 이른 연도',
    is_active           TINYINT      NOT NULL DEFAULT 1,
    first_collected_at  DATETIME     NOT NULL,
    last_collected_at   DATETIME     NOT NULL,
    created_at          TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at          TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (series_id),
    KEY idx_mineral (mineral_code),
    KEY idx_category (category_code)
) {_OPTS} COMMENT='KOMIS 자원 가격 시리즈 설명표 (STD-1)'
""",
f"""
CREATE TABLE IF NOT EXISTS {T_DATA} (
    series_id          VARCHAR(50) NOT NULL,
    period_end         DATE        NOT NULL COMMENT '기준일 (월 데이터는 월말일)',
    freq               CHAR(1)     NOT NULL,
    price              DOUBLE      NULL     COMMENT '기준가격',
    change_amt         DOUBLE      NULL     COMMENT '등락가',
    change_pct         DOUBLE      NULL     COMMENT '등락율(%)',
    first_collected_at DATETIME    NOT NULL COMMENT '최초 관측 시각(KST)',
    last_collected_at  DATETIME    NOT NULL COMMENT '최근 관측 시각(KST)',
    created_at         TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at         TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (series_id, period_end),
    KEY idx_period_end (period_end)
) {_OPTS} COMMENT='KOMIS 자원 가격 최신본 (STD-1)'
""",
f"""
CREATE TABLE IF NOT EXISTS {T_VIN} (
    id           BIGINT      NOT NULL AUTO_INCREMENT,
    series_id    VARCHAR(50) NOT NULL,
    period_end   DATE        NOT NULL,
    price        DOUBLE      NULL,
    change_type  VARCHAR(10) NOT NULL COMMENT 'new | revised',
    collected_at DATETIME    NOT NULL COMMENT '이 값을 관측한 시각(KST)',
    created_at   TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id),
    KEY idx_key_time (series_id, period_end, collected_at)
) {_OPTS} COMMENT='KOMIS 자원 가격 관측 이력 (STD-1 vintage)'
""",
]

with engine.begin() as cx:
    for ddl in DDL:
        cx.execute(text(ddl))
    # v3에서 만든 테이블에도 v4 컬럼 추가 (이미 있으면 건너뜀)
    cx.execute(text(f"ALTER TABLE {T_INFO} ADD COLUMN IF NOT EXISTS backfill_from_year SMALLINT NULL "
                    f"COMMENT '과거 백필을 시도한 가장 이른 연도' AFTER info_raw"))
print("테이블 준비 완료:", T_INFO, T_DATA, T_VIN)

테이블 준비 완료: kr_komis_mineral_price_info kr_komis_mineral_price kr_komis_mineral_price_vintage


## 4. 적재 함수

In [4]:
def now_kst():
    return datetime.now(KST).replace(tzinfo=None, microsecond=0)


def to_period_end(ymd, freq):
    """KOMIS 기준일(YYYYMMDD 또는 YYYYMM) → 기간 말일 date."""
    s = str(ymd).strip()
    if freq == "M" or len(s) == 6:
        return (pd.Timestamp(year=int(s[:4]), month=int(s[4:6]), day=1) + pd.offsets.MonthEnd(0)).date()
    return datetime.strptime(s[:8], "%Y%m%d").date()


def _find_unit(meta):
    """응답 원본에서 단위 후보 키를 찾아 반환 (없으면 None)."""
    for src in (meta.get("info_raw") or {}, meta.get("criteria_raw") or {}):
        for k, v in src.items():
            if v and any(t in k.lower() for t in ("unit", "untnm", "unt")):
                return str(v)[:50]
    return None


def _currency(unit):
    if not unit:
        return None
    u = unit.upper()
    if "$" in u or "USD" in u or "달러" in unit:
        return "USD"
    if "원" in unit or "KRW" in u:
        return "KRW"
    if "CNY" in u or "RMB" in u or "위안" in unit:
        return "CNY"
    return None


def _same(a, b):
    if a is None or b is None:
        return a is None and b is None
    return abs(float(a) - float(b)) <= 1e-9 * max(1.0, abs(float(a)))


def make_series_id(meta):
    crit = meta.get("price_criteria_code") or ""
    return f"{meta['mineral_code']}_{crit}" if crit else meta["mineral_code"]


def upsert_info(meta, freq, ts):
    unit = _find_unit(meta)
    rec = {
        "series_id": make_series_id(meta), "mineral_code": meta["mineral_code"],
        "mineral_name": meta["mineral_name"], "category_code": meta["category_code"],
        "category_name": meta["category_name"],
        "price_criteria_code": meta.get("price_criteria_code") or None,
        "price_criteria_name": meta.get("price_criteria_name"),
        "freq": freq, "unit": unit, "currency": _currency(unit),
        "criteria_raw": json.dumps(meta.get("criteria_raw") or {}, ensure_ascii=False),
        "info_raw": json.dumps(meta.get("info_raw") or {}, ensure_ascii=False),
        "ts": ts,
    }
    sql = f"""
    INSERT INTO {T_INFO}
        (series_id, mineral_code, mineral_name, category_code, category_name,
         price_criteria_code, price_criteria_name, freq, unit, currency,
         criteria_raw, info_raw, first_collected_at, last_collected_at)
    VALUES
        (:series_id, :mineral_code, :mineral_name, :category_code, :category_name,
         :price_criteria_code, :price_criteria_name, :freq, :unit, :currency,
         :criteria_raw, :info_raw, :ts, :ts)
    ON DUPLICATE KEY UPDATE
        mineral_name = VALUES(mineral_name), category_code = VALUES(category_code),
        category_name = VALUES(category_name), price_criteria_name = VALUES(price_criteria_name),
        freq = VALUES(freq), unit = COALESCE(VALUES(unit), unit),
        currency = COALESCE(VALUES(currency), currency),
        criteria_raw = VALUES(criteria_raw), info_raw = VALUES(info_raw),
        last_collected_at = VALUES(last_collected_at)
    """
    with engine.begin() as cx:
        cx.execute(text(sql), rec)
    return rec["series_id"]


def upsert_prices(series_id, df, freq, ts):
    """가격 저장. 신규 날짜나 값이 바뀐 날짜는 vintage에도 기록. 반환: (저장 건수, 이력 건수)"""
    recs = {}
    for ymd, p, c, cp in zip(df["crtrYmd"], df["price"], df["change"], df["change_pct"]):
        if pd.isna(p):
            continue
        pe = to_period_end(ymd, freq)
        recs[pe] = {"series_id": series_id, "period_end": pe, "freq": freq, "price": float(p),
                    "change_amt": None if pd.isna(c) else float(c),
                    "change_pct": None if pd.isna(cp) else float(cp), "ts": ts}
    recs = list(recs.values())
    if not recs:
        return 0, 0
    upsert = f"""
    INSERT INTO {T_DATA}
        (series_id, period_end, freq, price, change_amt, change_pct, first_collected_at, last_collected_at)
    VALUES
        (:series_id, :period_end, :freq, :price, :change_amt, :change_pct, :ts, :ts)
    ON DUPLICATE KEY UPDATE
        freq = VALUES(freq), price = VALUES(price),
        change_amt = VALUES(change_amt), change_pct = VALUES(change_pct),
        first_collected_at = LEAST(first_collected_at, VALUES(first_collected_at)),
        last_collected_at  = GREATEST(last_collected_at, VALUES(last_collected_at))
    """
    ins_vin = f"""
    INSERT INTO {T_VIN} (series_id, period_end, price, change_type, collected_at)
    VALUES (:series_id, :period_end, :price, :change_type, :ts)
    """
    with engine.begin() as cx:
        existing = {pe: v for pe, v in cx.execute(
            text(f"SELECT period_end, price FROM {T_DATA} WHERE series_id = :s"), {"s": series_id})}
        vin = []
        for r in recs:
            if r["period_end"] not in existing:
                vin.append(dict(r, change_type="new"))
            elif not _same(existing[r["period_end"]], r["price"]):
                vin.append(dict(r, change_type="revised"))
        cx.execute(text(upsert), recs)
        if vin:
            cx.execute(text(ins_vin), vin)
    return len(recs), len(vin)


def db_last_date(mineral_code):
    """DB에 저장된 해당 광종의 마지막 날짜 (없으면 None)."""
    q = text(f"SELECT MAX(d.period_end) FROM {T_DATA} d "
             f"JOIN {T_INFO} i ON d.series_id = i.series_id WHERE i.mineral_code = :m")
    with engine.connect() as cx:
        return cx.execute(q, {"m": mineral_code}).scalar()


def db_backfill_from(mineral_code):
    """이 광종에 대해 과거 백필을 시도한 가장 이른 연도 (없으면 None)."""
    q = text(f"SELECT MIN(backfill_from_year) FROM {T_INFO} WHERE mineral_code = :m")
    with engine.connect() as cx:
        return cx.execute(q, {"m": mineral_code}).scalar()


def mark_backfilled(mineral_code, year):
    q = text(f"UPDATE {T_INFO} SET backfill_from_year = LEAST(COALESCE(backfill_from_year, 9999), :y) "
             f"WHERE mineral_code = :m")
    with engine.begin() as cx:
        cx.execute(q, {"m": mineral_code, "y": int(year)})


def collect_to_db(end_year=None, mode="auto", backfill_start_year=2000,
                  categories=None, minerals=None, save_to_excel=False, output_dir=None):
    """KOMIS 가격을 받아 investar에 적재.

    mode
      "auto" : 광종별로 DB의 저장 기간을 보고 시작 연도를 자동 결정
               - DB에 없음                          → backfill_start_year부터 전체
               - backfill_start_year까지 백필한 적 없음 → 앞부분 보강 (한 번만)
               - 그 외                              → 마지막 저장 연도부터 갱신
               KOMIS가 그보다 늦게부터 제공하면 받은 만큼만 저장되고, 다음 실행부터는 갱신만 한다.
      "full" : 모든 광종을 backfill_start_year부터 다시 받음
    """
    end_year = end_year or date.today().year
    crawler = KOMISCrawler()
    ts = now_kst()
    summary = []
    for hp000 in (categories or list(crawler.categories)):
        cat = crawler.categories[hp000]
        print(f"\n===== {cat['name']} ({hp000}) =====")
        for name, code in cat["minerals"].items():
            if minerals and name not in minerals:
                continue
            try:
                last, done_from = db_last_date(code), db_backfill_from(code)
                if mode == "full" or last is None:
                    sy, why = backfill_start_year, "전체 백필"
                elif done_from is None or done_from > backfill_start_year:
                    sy, why = backfill_start_year, f"{backfill_start_year}년까지 백필 이력 없음 → 앞부분 보강"
                else:
                    sy, why = last.year, f"DB 마지막 {last} 이후 갱신"
                print(f"- {name}: {sy}~{end_year} ({why})")
                df, meta = crawler.fetch_series(name, hp000, sy, end_year)
                if df is None or df.empty:
                    print(f"[FAIL] {name}: 데이터 없음")
                    summary.append((cat["name"], name, None, None, None, 0, 0, "데이터 없음"))
                    continue
                freq = AVG_TO_FREQ[meta["avg_option"]]
                sid = upsert_info(meta, freq, ts)
                n, nv = upsert_prices(sid, df, freq, ts)
                if sy == backfill_start_year:
                    mark_backfilled(code, sy)
                if save_to_excel:
                    save_excel(df, meta, sy, end_year, output_dir)
                d0, d1 = df["crtrYmd"].min(), df["crtrYmd"].max()
                print(f"[OK]   {name:<6} {sid:<14} freq={freq}  수신 {d0}~{d1}, 저장 {n:>5}건, 신규/수정 {nv:>5}건")
                summary.append((cat["name"], name, sid, d0, d1, n, nv, "OK"))
            except Exception as e:
                print(f"[FAIL] {name}: {type(e).__name__}: {e}")
                summary.append((cat["name"], name, None, None, None, 0, 0, type(e).__name__))
            time.sleep(1)
        time.sleep(2)
    res = pd.DataFrame(summary, columns=["category", "mineral", "series_id", "recv_first", "recv_last",
                                         "n_saved", "n_new_or_revised", "status"])
    print(f"\n성공 {int((res.status == 'OK').sum())}/{len(res)}")
    return res


def load_komis_wide(category_code=None, start=None, value="price"):
    """DB에서 읽어 광종명을 컬럼으로 하는 표로 반환 (index = period_end)."""
    q = (f"SELECT d.period_end, i.mineral_name, d.{value} AS v FROM {T_DATA} d "
         f"JOIN {T_INFO} i ON d.series_id = i.series_id WHERE 1 = 1")
    params = {}
    if category_code:
        q += " AND i.category_code = :c"
        params["c"] = category_code
    if start:
        q += " AND d.period_end >= :st"
        params["st"] = start
    with engine.connect() as cx:
        long = pd.read_sql(text(q), cx, params=params)
    if long.empty:
        return long
    long["period_end"] = pd.to_datetime(long["period_end"])
    return long.pivot_table(index="period_end", columns="mineral_name", values="v", aggfunc="last").sort_index()

print("적재 함수 준비 완료")

적재 함수 준비 완료


## 5-0. (선택) KOMIS 제공 기간 진단
광종 하나를 골라 평균옵션(일/주/월)별로 몇 년부터 데이터가 있는지 확인합니다. DB에는 저장하지 않습니다.
1년씩 요청하므로 1~2분 걸립니다.

In [5]:
probe = probe_history("동", hp000="HP001", first_year=1990)
probe.groupby("avg_option", sort=False).agg(first_year=("year", "min"), last_year=("year", "max"), n_rows=("n_rows", "sum"))

동 (가격기준 501 LME CASH) 평균옵션별 제공 시작 연도:
  DAY  : 2002 ~ 2026  (6252건)
  WEEK : 2002 ~ 2026  (1303건)
  MONTH: 2002 ~ 2026  (297건)


,first_year,last_year,n_rows
avg_option,,,
DAY,2002,2026,6252
WEEK,2002,2026,1303
MONTH,2002,2026,297


## 5. 수집 및 적재
- `MODE = "auto"` (기본): 광종별로 DB에 있는 기간을 보고 알아서 시작 연도를 정합니다.
  DB에 없거나 앞부분이 비어 있으면 `BACKFILL_START_YEAR`부터 받고, 그 외에는 마지막 저장 연도부터 갱신합니다.
- `MODE = "full"`: 전 광종을 `BACKFILL_START_YEAR`부터 다시 받습니다.

In [6]:
MODE = "auto"                  # "auto" 또는 "full"
BACKFILL_START_YEAR = 2000     # KOMIS 제공 범위보다 이르면 해당 연도는 빈 응답으로 넘어감
SAVE_EXCEL = False             # True면 기존처럼 엑셀도 저장

result = collect_to_db(mode=MODE, backfill_start_year=BACKFILL_START_YEAR, save_to_excel=SAVE_EXCEL)
result


===== 비철금속 (HP001) =====
- 니켈: 2000~2026 (2000년까지 백필 이력 없음 → 앞부분 보강)
  니켈 DAY 연도별 수신: 2002:252, 2003:253, 2004:254, 2005:252, 2006:252, 2007:251, 2008:254, 2009:253, 2010:253, 2011:251, 2012:252, 2013:253, 2014:253, 2015:253, 2016:253, 2017:252, 2018:253, 2019:253, 2020:254, 2021:253, 2022:251, 2023:251, 2024:254, 2025:253, 2026:189
[OK]   니켈     MNRL0002_502   freq=D  수신 20020102~20260930, 저장  6252건, 신규/수정  5810건
- 동: 2000~2026 (2000년까지 백필 이력 없음 → 앞부분 보강)
  동 DAY 연도별 수신: 2002:252, 2003:253, 2004:254, 2005:252, 2006:252, 2007:251, 2008:254, 2009:253, 2010:253, 2011:251, 2012:252, 2013:253, 2014:253, 2015:253, 2016:253, 2017:252, 2018:253, 2019:253, 2020:254, 2021:253, 2022:251, 2023:251, 2024:254, 2025:253, 2026:189
[OK]   동      MNRL0008_501   freq=D  수신 20020102~20260930, 저장  6252건, 신규/수정  5810건
- 아연: 2000~2026 (2000년까지 백필 이력 없음 → 앞부분 보강)
  아연 DAY 연도별 수신: 2002:252, 2003:253, 2004:254, 2005:252, 2006:252, 2007:251, 2008:254, 2009:253, 2010:253, 2011:251, 2012:252, 2013:253, 2014:253,

,category,mineral,series_id,recv_first,recv_last,n_saved,n_new_or_revised,status
0,비철금속,니켈,MNRL0002_502,20020102,20260930,6252,5810,OK
1,비철금속,동,MNRL0008_501,20020102,20260930,6252,5810,OK
2,비철금속,아연,MNRL0023_561,20020102,20260930,6252,5810,OK
3,비철금속,알루미늄,MNRL0009_495,20020102,20260930,6252,5810,OK
4,비철금속,연,MNRL0022_499,20020102,20260930,6252,5810,OK
5,비철금속,주석,MNRL0016_493,20020102,20260930,6251,5809,OK
6,희소금속,리튬,MNRL0001_773,20180108,20260924,1937,1526,OK
7,희소금속,코발트,MNRL0003_791,20100702,20260924,3699,3299,OK
8,희소금속,망간,MNRL0004_815,20200910,20260924,1324,914,OK
9,희소금속,크롬,MNRL0021_794,20120712,20260924,3296,2885,OK


## 6. 적재 결과 검증

In [7]:
check_sql = f"""
SELECT i.category_name, i.mineral_name, i.series_id, i.price_criteria_name, i.freq, i.unit,
       COUNT(d.period_end)      AS n_obs,
       MIN(d.period_end)        AS first_date,
       MAX(d.period_end)        AS last_date,
       MAX(d.last_collected_at) AS last_collected_at
FROM {T_INFO} i
LEFT JOIN {T_DATA} d ON d.series_id = i.series_id
GROUP BY i.category_name, i.mineral_name, i.series_id, i.price_criteria_name, i.freq, i.unit
ORDER BY i.category_code, i.mineral_code
"""
with engine.connect() as cx:
    check = pd.read_sql(text(check_sql), cx)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", None)
check

,category_name,mineral_name,series_id,price_criteria_name,freq,unit,n_obs,first_date,last_date,last_collected_at
0,비철금속,니켈,MNRL0002_502,LME CASH,D,ton,6252,2002-01-02,2026-09-30,2026-10-01 15:55:56
1,비철금속,동,MNRL0008_501,LME CASH,D,ton,6252,2002-01-02,2026-09-30,2026-10-01 15:55:56
2,비철금속,알루미늄,MNRL0009_495,LME CASH,D,ton,6252,2002-01-02,2026-09-30,2026-10-01 15:55:56
3,비철금속,주석,MNRL0016_493,LME CASH,D,ton,6251,2002-01-02,2026-09-30,2026-10-01 15:55:56
4,비철금속,연,MNRL0022_499,LME CASH,D,ton,6252,2002-01-02,2026-09-30,2026-10-01 15:55:56
5,비철금속,아연,MNRL0023_561,LME CASH,D,ton,6252,2002-01-02,2026-09-30,2026-10-01 15:55:56
6,희소금속,리튬,MNRL0001_773,Lithium Carbonate,D,kg,1937,2018-01-08,2026-09-24,2026-10-01 15:55:56
7,희소금속,코발트,MNRL0003_791,Cobalt Metal,D,kg,3699,2010-07-02,2026-09-24,2026-10-01 15:55:56
8,희소금속,망간,MNRL0004_815,Ferro-manganese,D,mt,1324,2020-09-10,2026-09-24,2026-10-01 15:55:56
9,희소금속,흑연,MNRL0005_801,Flake Graphite,D,mt,2513,2015-08-31,2026-09-24,2026-10-01 15:55:56


## 7. 예시: DB에서 비철금속 가격 읽기

In [8]:
base = load_komis_wide(category_code="HP001", start="2024-01-01")
base.tail()

mineral_name,니켈,동,아연,알루미늄,연,주석
period_end,,,,,,
2026-09-24,16320.0,14765.0,3981.0,3228.0,1900.0,53875.0
2026-09-25,16050.0,14740.0,4060.0,3254.0,1902.5,54025.0
2026-09-28,16050.0,14544.5,3957.0,3248.5,1886.0,53750.0
2026-09-29,15860.0,14474.5,3977.0,3225.5,1876.0,53950.0
2026-09-30,15860.0,14487.0,3954.0,3204.0,1856.0,54075.0
